In [1]:
import faiss
import fitz

from sentence_transformers import CrossEncoder

from Config import Configs
from Config import ModelLoader as ML
from Libraries import Common_MyUtils as MU, Common_TextProcess as TP, Common_PdfProcess as PP
from Libraries import PDF_QualityCheck as QualityCheck, PDF_ExtractData as ExtractData, PDF_MergeData as MergeData
from Libraries import Json_ChunkUnder as ChunkUnder, Json_GetStructures as GetStructures, Json_ChunkMaster as ChunkMaster, Json_SchemaExt as SchemaExt
from Libraries import Faiss_Embedding as F_Embedding, Faiss_Searching as F_Searching, Faiss_ChunkMapping as chunkMapper
from Libraries import Summarizer_Runner as SummaryRun

e:\_DevTools\miniconda\envs\bruh\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## CONFIGURATION

#### HARD CODE

In [2]:
service = "Categories"
infilename = "HNMU"
jsonKey = "paragraphs"
jsonField = "Text"

MODEL_DIR = "Models"
MODEL_SUMARY = "Summarizer"
MODEL_ENCODE = "Sentence_Transformer"

#### LOAD CONFIG

In [3]:
config = Configs.ConfigValues(pdfname=infilename, service=service)

pdfPath = config["pdfPath"]
exceptPath = config["exceptPath"]
markerPath = config["markerPath"]
statusPath = config["statusPath"]

rawDataPath = config["rawDataPath"]
rawLvlsPath = config["rawLvlsPath"]
structsPath = config["structsPath"]
segmentPath = config["segmentPath"]
schemaPath = config["schemaPath"]
faissPath = config["faissPath"]
mappingPath = config["mappingPath"]
mapDataPath = config["mapDataPath"]
mapChunkPath = config["mapChunkPath"]
metaPath = config["metaPath"]

serviceSegmentPath = config["serviceSegmentPath"]
serviceFaissPath = config["serviceFaissPath"]
serviceMappingPath = config["serviceMappingPath"]
serviceMapDataPath = config["serviceMapDataPath"]
serviceMapChunkPath = config["serviceMapChunkPath"]
serviceMetaPath = config["serviceMetaPath"]

DATA_KEY = config["DATA_KEY"]
EMBE_KEY = config["EMBE_KEY"]
SEARCH_EGINE = config["SEARCH_EGINE"]
RERANK_MODEL = config["RERANK_MODEL"]
RESPON_MODEL = config["RESPON_MODEL"]
EMBEDD_MODEL = config["EMBEDD_MODEL"]
CHUNKS_MODEL = config["CHUNKS_MODEL"]
SUMARY_MODEL = config["SUMARY_MODEL"]
WORD_LIMIT = config["WORD_LIMIT"]

EMBEDD_CACHED_MODEL = f"{MODEL_DIR}/{MODEL_ENCODE}/{EMBEDD_MODEL}"
CHUNKS_CACHED_MODEL = F"{MODEL_DIR}/{MODEL_ENCODE}/{CHUNKS_MODEL}"
SUMARY_CACHED_MODEL = f"{MODEL_DIR}/{MODEL_SUMARY}/{SUMARY_MODEL}"

MAX_INPUT = 1024
MAX_TARGET = 256
MIN_TARGET = 64
TRAIN_EPOCHS = 3
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 4

## EXCEPTIONS

#### FUNCTIONS

In [4]:
def loadHardcodes(file_path, wanted=None):
    data = MU.readJson(file_path)
    if "items" not in data:
        return
    result = {}
    for item in data["items"]:
        key = item["key"]
        if (not wanted) or (key in wanted):
            result[key] = item["values"]
    return result

#### LOAD EXCEPTIONS

In [5]:
exceptData = loadHardcodes(exceptPath, wanted=["common_words", "proper_names", "abbreviations"])
markerData = loadHardcodes(markerPath, wanted=["keywords", "markers"])
statusData = loadHardcodes(statusPath, wanted=["brackets", "sentence_ends"])

## MODELS

#### CLASS

In [6]:
Loader = ML.ModelLoader()

#### LOAD MODELS

In [7]:
indexer, embeddDevice = Loader.loadEncoder(EMBEDD_MODEL, EMBEDD_CACHED_MODEL)
chunker, chunksDevice = Loader.loadEncoder(CHUNKS_MODEL, CHUNKS_CACHED_MODEL)

tokenizer, summarizer, summaryDevice = Loader.loadSummarizer(SUMARY_MODEL, SUMARY_CACHED_MODEL)

2026-01-09 21:08:19,258 - INFO - Load pretrained SentenceTransformer: Models/Sentence_Transformer/VoVanPhuc/sup-SimCSE-VietNamese-phobert-base
2026-01-09 21:08:19,259 - WARNING - You try to use a model that was created with version 5.1.0, however, your version is 3.0.1. This might cause unexpected behavior or errors. In that case, try to update to the latest version.






🔍 Loading SentenceTransformer (VoVanPhuc/sup-SimCSE-VietNamese-phobert-base) on cuda ...
CUDA supported: True
Number of GPUs: 1
Current GPU: NVIDIA GeForce RTX 2050
Capability: (8, 6)
CUDA version (PyTorch): 12.1
cuDNN version: 8907


2026-01-09 21:08:19,933 - INFO - 2 prompts are loaded, with the keys: ['query', 'document']
2026-01-09 21:08:19,935 - INFO - Load pretrained SentenceTransformer: Models/Sentence_Transformer/paraphrase-multilingual-MiniLM-L12-v2
2026-01-09 21:08:19,936 - WARNING - You try to use a model that was created with version 5.1.0, however, your version is 3.0.1. This might cause unexpected behavior or errors. In that case, try to update to the latest version.





📂 Loaded from cache: Models/Sentence_Transformer/VoVanPhuc/sup-SimCSE-VietNamese-phobert-base
✅ SentenceTransformer ready.

🔍 Loading SentenceTransformer (paraphrase-multilingual-MiniLM-L12-v2) on cuda ...
CUDA supported: True
Number of GPUs: 1
Current GPU: NVIDIA GeForce RTX 2050
Capability: (8, 6)
CUDA version (PyTorch): 12.1
cuDNN version: 8907


2026-01-09 21:08:20,891 - INFO - 2 prompts are loaded, with the keys: ['query', 'document']


📂 Loaded from cache: Models/Sentence_Transformer/paraphrase-multilingual-MiniLM-L12-v2
✅ SentenceTransformer ready.

🔍 Initializing summarizer (LongK171/bartpho-syllable-vnexpress) on cuda ...
CUDA supported: True
Number of GPUs: 1
Current GPU: NVIDIA GeForce RTX 2050
Capability: (8, 6)
CUDA version (PyTorch): 12.1
cuDNN version: 8907
📂 Loading summarizer from local cache...
✅ Summarizer ready on cuda


## MAIN FLOW CLASSES

#### EXTRACTOR

In [8]:
checker = QualityCheck.PDFQualityChecker()

dataExtractor = ExtractData.B1Extractor(
    exceptData,
    markerData,
    statusData,
    properNameMinCount=10
)

merger = MergeData.ParagraphMerger()

#### STRUCT CHUNKER

In [9]:
structAnalyzer = GetStructures.StructureAnalyzer(
    verbose=True
)

chunkBuilder = ChunkMaster.ChunkBuilder()

schemaExt = SchemaExt.JSONSchemaExtractor(
    listPolicy="first", 
    verbose=True
)

#### INDEXER

In [10]:
faissIndexer = F_Embedding.DirectFaissIndexer(
    indexer=indexer,
    device=str(embeddDevice),
    batch_size=32,
    show_progress=True,
    flatten_mode="split",
    join_sep="\n",
    allowed_schema_types=("string", "array", "dict"),
    max_chars_per_text=2000,
    normalize=True,
    verbose=False
)

#### SEGMENT CHUNKER

In [11]:
chunkUnder = ChunkUnder.ChunkUndertheseaBuilder(
    embedder=indexer,
    device=embeddDevice,
    minWords=256,
    maxWords=768,
    simThreshold=0.7,
    keySentRatio=0.4
)

#### SUMMARIZER

In [12]:
summaryEngine = SummaryRun.RecursiveSummarizer(
    tokenizer=tokenizer,
    summarizer=summarizer,
    sumDevice=summaryDevice,
    chunkBuilder=chunkUnder,
    maxLength=200,
    minLength=100,
    maxDepth=4
)

#### SEARCHER

In [13]:
reranker = CrossEncoder(RERANK_MODEL, device=str(embeddDevice))
searchEngine = F_Searching.SemanticSearchEngine(
    indexer=indexer,
    reranker=reranker,
    device=str(embeddDevice),
    normalize=True,
    topK=20,
    rerankK=10,
    rerankBatchSize=16
)

## MAIN FLOW FUNCTIONS

### PREPROCESS

#### CHECKER

In [14]:
def pdfCheck(pdfDoc):
    isGood, metrics = checker.evaluate(pdfDoc)
    return isGood, metrics

#### EXTRACTOR

In [15]:
def extractRun(pdfDoc):
    extractedData = dataExtractor.extract(pdfDoc)
    rawDataDict = merger.merge(extractedData)
    return rawDataDict

### PROCESS FOR SEARCHING

#### STRUCT GETTER

In [16]:
def structRun(rawDataDict):
    markers =       structAnalyzer.extractMarkers(rawDataDict)
    structures =    structAnalyzer.buildStructures(markers)
    dedup =         structAnalyzer.deduplicate(structures)
    top =           structAnalyzer.selectTop(dedup)
    RawLvlsDict =   structAnalyzer.extendTop(top, dedup)
    
    print(MU.jsonConvert(RawLvlsDict, pretty=True))
    return RawLvlsDict

#### STRUCT CHUNKER

In [17]:
def chunkRun(RawLvlsDict=None, rawDataDict=None):
    StructsDict = chunkBuilder.build(RawLvlsDict, rawDataDict)
    return StructsDict

#### SEGMENT CHUNKER

In [18]:
def SegmentRun(StructsDict, RawLvlsDict):
    firstKey = list(RawLvlsDict[0].keys())[0]

    segmentDict = []
    for item in StructsDict:
        value = item.get(firstKey)
        if not value:
            continue
        
        if isinstance(value, list):
            value = " ".join(
                v.strip() for v in value
                if isinstance(v, str) and v.strip().lower() != "none"
            )
            if value.strip():
                segmentDict.append(item)

        elif isinstance(value, str):
            text = value.strip()
            if text and text.lower() != "none":
                segmentDict.append(item)

    for i, item in enumerate(segmentDict, start=1):
        item["Index"] = i

    return segmentDict


#### SCHEMA GETTER

In [19]:
def schemaRun(segmentDict):
    schemaDict = schemaExt.schemaRun(segmentDict=segmentDict)
    print(schemaDict)
    return schemaDict

#### INDEXER

In [20]:
def Indexing(schemaDict):
    faissIndex, mapping, mapData, chunkGroups = faissIndexer.buildFromJson(
        segmentPath=segmentPath,
        schemaDict=schemaDict,
        faissPath=faissPath,
        mapDataPath=mapDataPath,
        mappingPath=mappingPath,
        mapChunkPath=mapChunkPath
    )
    return faissIndex, mapping, mapData, chunkGroups

### PROCESS FOR CLASSIFICATION

#### RAW MERGER

In [21]:
def mergebyText(rawDataDict):
    mergedText = TP.mergeTxt(rawDataDict, jsonKey, jsonField)
    return mergedText

#### SUMMARIZER

In [22]:
def summaryRun(mergedText):
    summarized = summaryEngine.summarize(mergedText, minInput = 256, maxInput = 1024)
    return summarized

### FINAL PROCESS

#### SEARCHER

In [23]:
def runSearch(query, faissIndex, mapping, mapData, mapChunk):
    results = searchEngine.search(
        query=query,
        faissIndex=faissIndex,
        mapping=mapping,
        mapData=mapData,
        mapChunk=mapChunk,
        topK=20
    )
    return results

#### RERANKER

In [24]:
def runRerank(query, results):
    reranked = searchEngine.rerank(
        query=query,
        results=results,
        topK=10
    )
    return reranked

## MERGED FUNCTIONS

#### READ DATA

In [25]:
def ReadData(segmentPath, faissPath, mappingPath, mapDataPath, mapChunkPath):
    segmentDict = MU.readJson(segmentPath)
    faissIndex = faiss.read_index(faissPath)
    mapping = MU.readJson(mappingPath)
    mapData = MU.readJson(mapDataPath)
    mapChunk = MU.readJson(mapChunkPath)
    return {
        "segmentDict": segmentDict,
        "faissIndex": faissIndex,
        "mapping": mapping,
        "mapData": mapData,
        "mapChunk": mapChunk
    }

#### READ PDF

In [26]:
def preReadPDF(pdfPath=None, pdfBytes=None):
    if pdfBytes is not None:
        pdfDoc = fitz.open(stream=pdfBytes, filetype="pdf")
    elif pdfPath is not None:
        pdfDoc = fitz.open(pdfPath)
    else:
        return None
    
    checker = QualityCheck.PDFQualityChecker()
    is_good, info = checker.evaluate(pdfDoc)
    print(info)
    if is_good:
        print("[PASS] Tiếp tục xử lý.")
    else:
        print("[DENY] Bỏ qua file này.")
        pdfDoc.close()
        return None
        
    rawDataDict = extractRun(pdfDoc)
    MU.writeJson(rawDataDict, rawDataPath, indent=1)
    pdfDoc.close()
    
    return rawDataDict

#### PREPARE DATA

In [27]:
def PrepareData(segmentPath, faissPath, mappingPath, mapDataPath, mapChunkPath, rawDataDict=None):            
    if rawDataDict is not None:
        RawLvlsDict = structRun(rawDataDict)
        MU.writeJson(RawLvlsDict, rawLvlsPath, indent=2)

        StructsDict = chunkRun(RawLvlsDict, rawDataDict)
        MU.writeJson(StructsDict, structsPath, indent=2)

        segmentDict = SegmentRun(StructsDict, RawLvlsDict)
        MU.writeJson(segmentDict, segmentPath, indent=2)
        
    elif MU.fileExists(segmentPath):
        segmentDict = MU.readJson(segmentPath)
        
    else:
        return {
            "segmentDict": None,
            "faissIndex": None,
            "mapping": None,
            "mapData": None,
            "mapChunk": None
        }

    
    schemaDict = schemaRun(segmentDict)
    MU.writeJson(schemaDict, schemaPath, indent=2)

    faissIndex, mapping, mapData, chunkGroups = Indexing(schemaDict)
    MU.writeJson(mapping, mappingPath, indent=2)
    MU.writeJson(mapData, mapDataPath, indent=2)
    
    faiss.write_index(faissIndex, faissPath)
    MU.writeChunkmap(mapChunkPath, segmentPath, chunkGroups)
    mapChunk = MU.readJson(mapChunkPath)
    
    print("\nCompleted!")
    
    return {
        "segmentDict": segmentDict,
        "faissIndex": faissIndex,
        "mapping": mapping,
        "mapData": mapData,
        "mapChunk": mapChunk
    }

#### SUMMARIZE

In [28]:
def summarizeDcmt(rawDataDict):
    mergedText = mergebyText(rawDataDict)
    summarized = summaryRun(mergedText)
    return summarized["summaryText"]

#### CLASSIFY

In [29]:
def classifyDocument(summaryText):
    readedData = ReadData(serviceSegmentPath, serviceFaissPath, serviceMappingPath, serviceMapDataPath, serviceMapChunkPath)
    servicesegmentDict = readedData.get("segmentDict")
    servicefaissIndex = readedData.get("faissIndex")
    servicemapping = readedData.get("mapping")
    servicemapData = readedData.get("mapData")
    servicemapChunk = readedData.get("mapChunk")
    
    resuls = runSearch(summaryText, servicefaissIndex, servicemapping, servicemapData, servicemapChunk)
    reranked = runRerank(summaryText, resuls)
    
    bestCategory = chunkMapper.processChunksPipeline(rerankedResults=reranked, segmentDict=servicesegmentDict, dropFields=["Index"], fields=["Article"], nChunks=1)
    bestArticles = [item["fields"].get("Article") for item in bestCategory["extractedFields"]]
    bestArticle = bestArticles[0] if len(bestArticles) == 1 else ", ".join(bestArticles)
    return bestArticle

## MAIN

In [ ]:
def mainRun(pdfPath=None, pdfBytes=None, queryText = "Nhiệm vụ của sinh viên là gì?"):
    # summaryText = ""
    # bestArticle = ""
    # response = ""
    
    rawDataDict = preReadPDF(pdfPath, pdfBytes)
    
    # summaryText = summarizeDcmt(rawDataDict)
    # bestArticle = classifyDocument(summaryText)
    
    preparedData = PrepareData(segmentPath, faissPath, mappingPath, mapDataPath, mapChunkPath, rawDataDict)
    # preparedData = PrepareData(segmentPath, faissPath, mappingPath, mapDataPath, mapChunkPath)
    segmentDict=preparedData.get("segmentDict")
    faissIndex=preparedData.get("faissIndex")
    mapping=preparedData.get("mapping")
    mapData=preparedData.get("mapData")
    mapChunk=preparedData.get("mapChunk")
    
    readedData = ReadData(segmentPath, faissPath, mappingPath, mapDataPath, mapChunkPath)
    segmentDict = readedData.get("segmentDict")
    faissIndex = readedData.get("faissIndex")
    mapping = readedData.get("mapping")
    mapData = readedData.get("mapData")
    mapChunk = readedData.get("mapChunk")
    
    resuls = runSearch(queryText, faissIndex, mapping, mapData, mapChunk)
    reranked = runRerank(queryText, resuls)
    chunkReturn = chunkMapper.processChunksPipeline(
        reranked=reranked,
        segmentDict=segmentDict,
        dropFields=["Index"],          # 1) Trường bị bỏ qua (áp dụng toàn bộ). None → không bỏ
        fields=["Article"],            # 2) Trường muốn trả cho mỗi chunk. None → tất cả top-level còn lại
        nChunks=1,                     # 3) Số lượng chunk gốc được trả về. None → tất cả
    )
    print(chunkReturn["chunksText"])

In [31]:
queryText = "Sinh viên được xét và công nhận tốt nghiệp khi có đủ các điều kiện nào?"
mainRun(pdfPath=pdfPath, pdfBytes=None, queryText=queryText)

{'checkMess': '✅ Đạt yêu cầu', 'totalChars': 64470, 'invalidRatio': 0.002, 'whitespaceRatio': 0.0, 'shortLineRatio': 0.027}
[PASS] Tiếp tục xử lý.
[12->13] Merge=False | Reason: Fallback
[14->15] Merge=False | Reason: Fallback
[35->36] Merge=False | Reason: Fallback
[930->931] Merge=False | Reason: Fallback
[950->951] Merge=False | Reason: Fallback
[953->954] Merge=False | Reason: Fallback
[
  {
    "Level 1": [
      "Chương XVI"
    ],
    "Level 2": [
      "Điều 123. "
    ],
    "Article": [
      "123. "
    ],
    "Content": [
      "abc) ",
      "none"
    ]
  }
]
{'Index': 'number', 'Level 1': 'string', 'Level 2': 'string', 'Article': 'string', 'Content': 'array'}


Batches:   0%|          | 0/13 [00:00<?, ?it/s]


Completed!


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 KẾT QUẢ SEARCH:

[{'index': 279, 'key': 'Article', 'text': '1. Sinh viên được xét và công nhận tốt nghiệp khi có đủ các điều kiện sau:', 'faissScore': 0.9191442728042603, 'chunkIds': [111]}, {'index': 314, 'key': 'Article', 'text': '2. Sinh viên được xem xét chuyển trường khi có đủ các điều kiện sau:', 'faissScore': 0.7291363477706909, 'chunkIds': [127]}, {'index': 315, 'key': 'Content[1]', 'text': 'b) Sinh viên đạt điều kiện trúng tuyển của chương trình, ngành đào tạo cùng khóa tuyển sinh tại nơi chuyển đến;', 'faissScore': 0.7130061388015747, 'chunkIds': [127]}, {'index': 285, 'key': 'Article', 'text': '2. Những sinh viên đủ điều kiện tốt nghiệp đã hoàn thành nghĩa vụ tài chính với Trường được Hiệu trưởng cấp bằng tốt nghiệp trong thời hạn 01 tháng từ thời điểm xét tốt nghiệp.', 'faissScore': 0.7121524214744568, 'chunkIds': [112]}, {'index': 208, 'key': 'Article', 'text': '4. Sinh viên có nhu cầu có thể đăng kí học bổ sung cho đủ khối lượng học tập tối thiểu được quy định tại khoản

In [32]:
queries = """
Sinh viên được xét và công nhận tốt nghiệp khi có đủ các điều kiện nào?
Cách hủy học phần đã đăng kí?
Tôi có thể được bảo lưu kết quả học khi nào?
Tôi có thể chuyển ngành đào đạo khi nào?
Sinh viên có quyền gì?
"""